In [11]:
import pandas as pd
import os

ROOT = "/Users/marouanaitslimani/Desktop/uni_projects/ProjetBDDAvance"
os.chdir(ROOT)

# exploration de nos données pour voir quelles colonnes on a 

sources = {
    "accidents_fr_caract":   "data/raw/accidents_fr/caracteristiques_2021.csv",
    "accidents_fr_lieux":    "data/raw/accidents_fr/lieux_2021.csv",
    "accidents_fr_usagers":  "data/raw/accidents_fr/usagers_2021.csv",
    "accidents_fr_vehicules":"data/raw/accidents_fr/vehicules_2021.csv",
    "accidents_uk_collision":"data/raw/accidents_uk/dft-road-casualty-statistics-collision-2021.csv",
    "accidents_uk_casualty": "data/raw/accidents_uk/dft-road-casualty-statistics-casualty-2021.csv",
    "accidents_uk_vehicle":  "data/raw/accidents_uk/dft-road-casualty-statistics-vehicle-2021.csv",
    "meteo_fr":              "data/raw/meteo_fr/meteo_fr_2021.csv",
    "meteo_uk":              "data/raw/meteo_uk/meteo_uk_2021.csv",
}

for nom, chemin in sources.items():
    print(f"SOURCE : {nom}")
    try:
        df = pd.read_csv(chemin, nrows=3, sep=None, engine="python")
        print(f"Colonnes ({len(df.columns)}) : {list(df.columns)}")
        print(f"Exemple ligne 1 :")
        print(df.iloc[0].to_dict())
    except Exception as e:
        print(f"ERREUR : {e}")
    print()

SOURCE : accidents_fr_caract
Colonnes (15) : ['Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg', 'int', 'atm', 'col', 'adr', 'lat', 'long']
Exemple ligne 1 :
{'Num_Acc': 202100000001, 'jour': 30, 'mois': 11, 'an': 2021, 'hrmn': '07:32', 'lum': 2, 'dep': 30, 'com': 30319, 'agg': 1, 'int': 1, 'atm': 1, 'col': 1, 'adr': 'CD 981', 'lat': '44,0389580000', 'long': '4,3480220000'}

SOURCE : accidents_fr_lieux
Colonnes (18) : ['Num_Acc', 'catr', 'voie', 'v1', 'v2', 'circ', 'nbv', 'vosp', 'prof', 'pr', 'pr1', 'plan', 'lartpc', 'larrout', 'surf', 'infra', 'situ', 'vma']
Exemple ligne 1 :
{'Num_Acc': 202100000001, 'catr': 3, 'voie': '981', 'v1': -1, 'v2': nan, 'circ': 2, 'nbv': 2, 'vosp': 0, 'prof': 1, 'pr': '(1)', 'pr1': '(1)', 'plan': 1, 'lartpc': nan, 'larrout': -1, 'surf': 1, 'infra': 0, 'situ': 1, 'vma': 80}

SOURCE : accidents_fr_usagers
Colonnes (16) : ['Num_Acc', 'id_usager', 'id_vehicule', 'num_veh', 'place', 'catu', 'grav', 'sexe', 'an_nais', 'trajet', 'secu1', 'secu2'

In [12]:
# DIM_PAYS
dim_pays = pd.DataFrame([
    {"id_pays": 1, "code_pays": "FR", "nom_pays": "France"},
    {"id_pays": 2, "code_pays": "UK", "nom_pays": "Royaume-Uni"},
])

os.makedirs("data/processed", exist_ok=True)
dim_pays.to_csv("data/processed/DIM_PAYS.csv", index=False)
print("DIM_PAYS :")
print(dim_pays)

DIM_PAYS :
   id_pays code_pays     nom_pays
0        1        FR       France
1        2        UK  Royaume-Uni


In [6]:
!pip install holidays --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.2 MB/s  0:00:00


In [8]:
# DIM_TEMPS
import holidays

ANNEES = [2005, 2007, 2009, 2011, 2013, 2015, 2017, 2019, 2021]

jours_feries_fr = holidays.France(years=ANNEES)
jours_feries_uk = holidays.UnitedKingdom(years=ANNEES)

JOURS_FR = {
    0: "Lundi", 1: "Mardi", 2: "Mercredi", 3: "Jeudi",
    4: "Vendredi", 5: "Samedi", 6: "Dimanche"
}
MOIS_FR = {
    1: "Janvier", 2: "Février", 3: "Mars", 4: "Avril",
    5: "Mai", 6: "Juin", 7: "Juillet", 8: "Août",
    9: "Septembre", 10: "Octobre", 11: "Novembre", 12: "Décembre"
}

rows = []
id_temps = 1

for annee in ANNEES:
    for mois in range(1, 13):
        for jour in range(1, 32):
            try:
                date = pd.Timestamp(year=annee, month=mois, day=jour)
            except ValueError:
                continue

            rows.append({
                "id_temps":     id_temps,
                "date":         date.strftime("%Y-%m-%d"),
                "jour":         date.day,
                "mois":         date.month,
                "annee":        date.year,
                "trimestre":    date.quarter,
                "jour_semaine": date.dayofweek + 1,
                "nom_jour":     JOURS_FR[date.dayofweek],
                "nom_mois":     MOIS_FR[date.month],
                "est_weekend":  int(date.dayofweek >= 5),
                "est_ferie_fr": int(date in jours_feries_fr),
                "est_ferie_uk": int(date in jours_feries_uk),
                "saison":       (
                    "Hiver"     if mois in [12, 1, 2] else
                    "Printemps" if mois in [3, 4, 5]  else
                    "Ete"       if mois in [6, 7, 8]  else
                    "Automne"
                ),
            })
            id_temps += 1

dim_temps = pd.DataFrame(rows)
dim_temps.to_csv("data/processed/DIM_TEMPS.csv", index=False)
print(f"DIM_TEMPS générée : {len(dim_temps)} lignes")
print(dim_temps.head(3))

DIM_TEMPS générée : 3285 lignes
   id_temps        date  jour  mois  annee  trimestre  jour_semaine  nom_jour  \
0         1  2005-01-01     1     1   2005          1             6    Samedi   
1         2  2005-01-02     2     1   2005          1             7  Dimanche   
2         3  2005-01-03     3     1   2005          1             1     Lundi   

  nom_mois  est_weekend  est_ferie_fr  est_ferie_uk saison  
0  Janvier            1             1             1  Hiver  
1  Janvier            1             0             0  Hiver  
2  Janvier            0             0             1  Hiver  
